<a href="https://colab.research.google.com/github/Akyzia/Centroides_Poligonos/blob/main/Centroides_2x2m_grids.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

CONNECT TO GOOGLE DRIVE

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

IMPORT LIBRARIES

In [ ]:
import geopandas as gpd
from shapely.geometry import Polygon, Point
import matplotlib.pyplot as plt

HERE I GENERATE THE 2M X 2M GRID THAT INTERSECTS WITH THE BUILDING FEATURES, AND THEN I EXTRACT THE CENTROIDS OF THESE GRIDS
HOWEVER, I EXCLUDE THE CENTROIDS THAT DO NOT HAVE AT LEAST 2M OF DISTANCE FROM ANOTHER CENTROID

In [ ]:
# Function to generate a grid of cells within a polygon
def generate_grid_within_bounds(bounds, grid_size):
    minx, miny, maxx, maxy = bounds  # Polygon boundaries
    cells = []
    x = minx
    while x < maxx:
        y = miny
        while y < maxy:
            # Create a square cell of grid_size x grid_size
            cell = Polygon([(x, y), (x + grid_size, y),
                            (x + grid_size, y + grid_size), (x, y + grid_size)])
            cells.append(cell)  # Add complete cell
            y += grid_size
        x += grid_size
    return cells

# Function to check the minimum distance of 2m between centroids
def is_valid_centroid(new_centroid, existing_centroids, min_distance):
    for centroid in existing_centroids:
        if new_centroid.distance(centroid) < min_distance:
            return False
    return True

# Path to the shapefile in Google Drive
filepath = '/content/drive/MyDrive/Centroides_AUC_DSA_MuncipioX/Edif_Const_pol.shp' # Your building/construction shapefile in polygon format (save it in your drive)

# Load polygon layer data (buildings)
gdf = gpd.read_file(filepath)

# Define grid size (2m x 2m) and minimum distance
grid_size = 2.0
min_distance = 2.0  # Minimum distance between centroids

# Store all generated grids and cell centroids, without duplication
grids = []
centroids = []

# Use a set to track grids that have already been added
grid_set = set()

# Iterate over all polygon features (buildings)
for _, row in gdf.iterrows():
    polygon = row['geometry']  # Building polygon
    bounds = polygon.bounds  # Polygon boundaries
    cells = generate_grid_within_bounds(bounds, grid_size)  # Generate complete grid cells

    # Filter cells that intersect the polygon
    intersecting_cells = [cell for cell in cells if cell.intersects(polygon)]

    for cell in intersecting_cells:
        # Check if the grid has already been processed, avoiding overlap
        if tuple(cell.exterior.coords) not in grid_set:
            grid_set.add(tuple(cell.exterior.coords))  # Add unique grid to the set
            new_centroid = cell.centroid  # New centroid

            # Check if the minimum distance is respected
            if is_valid_centroid(new_centroid, centroids, min_distance):
                grids.append(cell)  # Store unique grid
                centroids.append(new_centroid)  # Store valid centroid

# Convert the list of grids and centroids to GeoDataFrames
grids_gdf = gpd.GeoDataFrame(geometry=grids, crs=gdf.crs)
centroids_gdf = gpd.GeoDataFrame(geometry=centroids, crs=gdf.crs)

# Save the generated grids and centroids as shapefiles
output_filepath_grids = '/content/drive/MyDrive/Centroides_AUC_DSA_MunicipioX/Edificacoes_polig_grids_complete.shp' # Your resulting grid (will be saved in the corresponding folder)
output_filepath_centroids = '/content/drive/MyDrive/Centroides_AUC_DSA_MunicipioX/Edificacoes_polig_grid_centroids_complete.shp' # Your resulting centroids (will be saved in the corresponding folder)

grids_gdf.to_file(output_filepath_grids)  # Save grids
centroids_gdf.to_file(output_filepath_centroids)  # Save centroids

# Plot the results (optional)
ax = gdf.plot(edgecolor='black', facecolor='none', figsize=(8, 8))  # Building polygons
grids_gdf.plot(ax=ax, edgecolor='blue', facecolor='none')  # Complete grids
centroids_gdf.plot(ax=ax, color='red', markersize=2)  # Grid centroids
plt.show()